# BrNPDB-GNN: Brazilian Natural Products Graph Neural Network

This notebook demonstrates the GNN project for analyzing Brazilian natural products and their bioactivities.

In [ ]:
import os
# Disable CUDA to avoid issues
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['TORCH_USE_CUDA_DSA'] = '0'

import torch
import torch_geometric
import numpy as np
import pandas as pd
from torch_geometric.loader import DataLoader

print("PyTorch version:", torch.__version__)
print("PyTorch Geometric version:", torch_geometric.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Using device: cpu")

## Load Data

Let's load some of the processed molecular graphs.

In [ ]:
# Load a few Tox21 graphs
import os
tox21_dir = "data/processed/graphs_tox21"
tox21_files = sorted(os.listdir(tox21_dir))[:5]  # Load first 5

tox21_graphs = []
for fname in tox21_files:
    if fname.endswith('.pt'):
        data = torch.load(os.path.join(tox21_dir, fname))
        tox21_graphs.append(data)

print(f"Loaded {len(tox21_graphs)} Tox21 graphs")
print(f"Sample graph - Nodes: {tox21_graphs[0].x.shape[0]}, Edges: {tox21_graphs[0].edge_index.shape[1]}")

## Load Pre-trained Model

Load the pre-trained GNN encoder.

In [ ]:
from src.models.gin_model import GINEEncoder

# Model parameters
num_node_feats = 35
num_edge_feats = 9

# Load pre-trained encoder
encoder = GINEEncoder(
    in_channels=num_node_feats,
    edge_dim=num_edge_feats, 
    hidden_channels=128, 
    num_layers=4, 
    dropout=0.2
)

# Load trained weights
encoder.load_state_dict(torch.load("models/checkpoints/tox21/best_encoder.pt", map_location="cpu"))
encoder.eval()

print("✅ Pre-trained encoder loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in encoder.parameters()):,}")

## Extract Molecular Embeddings

Use the encoder to extract embeddings from the molecular graphs.

In [ ]:
def extract_embeddings_simple(encoder, graphs, device='cpu'):
    """Extract embeddings from a list of graphs"""
    encoder.eval()
    embeddings = []
    
    with torch.no_grad():
        for data in graphs:
            # Forward pass
            emb = encoder(data.x, data.edge_index, data.edge_attr, torch.tensor([data.num_nodes]))
            # Global mean pooling
            emb = emb.mean(dim=0)
            embeddings.append(emb.numpy())
    
    return np.array(embeddings)

# Extract embeddings
embeddings = extract_embeddings_simple(encoder, tox21_graphs)

print(f"Extracted embeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Sample embedding (first 10 values): {embeddings[0][:10]}")

## Compute Similarity

Compute cosine similarity between molecular embeddings.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

# Normalize embeddings for cosine similarity
emb_norm = normalize(embeddings, norm='l2')

# Compute similarity matrix
similarity_matrix = cosine_similarity(emb_norm)

print(f"Similarity matrix shape: {similarity_matrix.shape}")
print("Sample similarity scores:")
for i in range(min(3, len(similarity_matrix))):
    for j in range(min(3, len(similarity_matrix[i]))):
        if i != j:
            print(f"  Molecule {i} vs {j}: {similarity_matrix[i,j]:.3f}")

## Summary

✅ Successfully demonstrated:
- Loading molecular graph data
- Loading pre-trained GNN model
- Extracting molecular embeddings
- Computing molecular similarities

The project is working! You can now:
1. Run the full evaluation script (may need CUDA fixes)
2. Train new models on different datasets
3. Analyze molecular similarities and bioactivities